# Day 10 — Async Python
### Vidya Sankalp | Python for AI Engineering

**Goal:** Learn how Python can handle multiple slow I/O operations efficiently.

By the end of this notebook, you will understand:
- Why sequential waiting can be slow
- `async def`, `await`, and `asyncio.run()`
- Running independent tasks concurrently with `asyncio.gather()`
- Handling failures and timeouts
- Calling APIs asynchronously with HTTPX
- Validating API data with Pydantic

> We will start with simple examples that do not need an API key or database. The external API section requires an internet connection.

## 1. Why do we need async?

Imagine that an application needs information about three customers. Each request takes one second.

In a sequential program, we wait for customer 1, then customer 2, then customer 3. If each takes one second, the total waiting time is roughly three seconds.

With asynchronous programming, we can start all three independent requests and let their waiting time overlap.

**Important:** Async is especially useful for I/O-bound work—such as waiting for APIs, databases, or files. It does not automatically speed up CPU-heavy calculations.

In [ ]:
import time

def get_customer(customer_id):
    # Simulate waiting for an external service
    time.sleep(1)
    return f"Customer {customer_id}"

start = time.perf_counter()

customers = [
    get_customer(1),
    get_customer(2),
    get_customer(3),
]

print(customers)
print(f"Sequential time: {time.perf_counter() - start:.2f} seconds")

The example above blocks while each call waits. Now let's use `asyncio.sleep()` to simulate the same waiting without blocking the event loop.

## 2. Three important building blocks

| Syntax | Plain-English meaning |
|---|---|
| `async def` | Defines an asynchronous function (a coroutine function). |
| `await` | Waits for an asynchronous operation to finish while allowing other tasks to make progress. |
| `asyncio.run()` | Starts and manages an event loop for a simple async program. |

Think of `await` as: **“I need this result, but while I'm waiting, other async work can progress.”**

In [ ]:
import asyncio
import time

async def get_customer_async(customer_id):
    await asyncio.sleep(1)  # Simulate an async network wait
    return f"Customer {customer_id}"

async def main():
    customer = await get_customer_async(1)
    print(customer)

asyncio.run(main())

## 3. Run independent tasks concurrently with `asyncio.gather()`

`asyncio.gather()` lets us wait for multiple async operations together.

Use it when the tasks are independent—for example, fetching customer, order, and product data from separate services.

In [ ]:
async def main():
    start = time.perf_counter()

    results = await asyncio.gather(
        get_customer_async(1),
        get_customer_async(2),
        get_customer_async(3),
    )

    print(results)
    print(f"Concurrent time: {time.perf_counter() - start:.2f} seconds")

asyncio.run(main())

**What to notice:** Each task still waits for one second, but the waits overlap. The total time should be around one second rather than three (small differences are normal).

`gather()` is not magic: if tasks depend on one another, you may need to run them in sequence.

## 4. Handle errors and timeouts

Real services can fail. A request may time out, return an error, or be unavailable.

First, let's see how one async task can raise an exception.

In [ ]:
async def get_order(order_id):
    await asyncio.sleep(0.2)

    if order_id == 2:
        raise ValueError("Order 2 was not found")

    return {"order_id": order_id, "status": "shipped"}

async def main():
    try:
        order = await get_order(1)
        print(order)
    except ValueError as error:
        print(f"Could not fetch order: {error}")

asyncio.run(main())

When using `asyncio.gather()`, an exception can cause the awaited call to raise. For a beginner-friendly pattern, handle errors inside each task when you want to collect both successful results and readable error messages.

In [ ]:
async def safe_get_order(order_id):
    try:
        return await get_order(order_id)
    except ValueError as error:
        return {"order_id": order_id, "error": str(error)}

async def main():
    results = await asyncio.gather(
        safe_get_order(1),
        safe_get_order(2),
        safe_get_order(3),
    )
    for result in results:
        print(result)

asyncio.run(main())

### Add a timeout

`asyncio.wait_for()` stops waiting for an operation after the specified number of seconds. In real API calls, also configure the HTTP client's own timeout.

In [ ]:
async def slow_operation():
    await asyncio.sleep(3)
    return "Finished"

async def main():
    try:
        result = await asyncio.wait_for(slow_operation(), timeout=1)
        print(result)
    except asyncio.TimeoutError:
        print("The operation took too long.")

asyncio.run(main())

## 5. Real API calls with HTTPX (requires internet)

HTTPX provides `AsyncClient` for asynchronous HTTP requests. It is similar to the regular HTTP client introduced in Day 09, but its network calls are awaitable.

If HTTPX is not installed in your environment, run the next cell once.

In [ ]:
# Uncomment and run only if HTTPX is not installed:
# %pip install httpx

In [ ]:
import asyncio
import httpx

BASE_URL = "https://dummyjson.com"

async def fetch_product(client, product_id):
    response = await client.get(f"{BASE_URL}/products/{product_id}", timeout=10.0)
    response.raise_for_status()  # Raise an error for unsuccessful HTTP status codes
    return response.json()

async def main():
    async with httpx.AsyncClient() as client:
        products = await asyncio.gather(
            fetch_product(client, 1),
            fetch_product(client, 2),
            fetch_product(client, 3),
        )

    for product in products:
        print(product["id"], product["title"])

# Run this cell when you have an internet connection.
asyncio.run(main())

### What happened?

1. We created one `AsyncClient` and reused it.
2. Each `client.get()` call is preceded by `await`.
3. `asyncio.gather()` runs the independent requests concurrently.
4. `raise_for_status()` makes HTTP errors visible instead of silently treating them as success.
5. The response body is JSON, which Python converts into dictionaries and lists.

In a notebook environment that already has an active event loop, use `await main()` instead of `asyncio.run(main())`.

## 6. Validate API data with Pydantic

An API response is external data. Before relying on it in application logic, we can validate its structure with Pydantic.

This section connects **Day 08 (Pydantic)** and **Day 09 (HTTP requests)** with async Python.

In [ ]:
# Uncomment and run only if Pydantic is not installed:
# %pip install pydantic

In [ ]:
from pydantic import BaseModel

class Product(BaseModel):
    id: int
    title: str
    price: float

async def fetch_validated_product(client, product_id):
    response = await client.get(f"{BASE_URL}/products/{product_id}", timeout=10.0)
    response.raise_for_status()

    # Validate the JSON dictionary against the Product model
    return Product.model_validate(response.json())

async def main():
    async with httpx.AsyncClient() as client:
        products = await asyncio.gather(
            fetch_validated_product(client, 1),
            fetch_validated_product(client, 2),
            fetch_validated_product(client, 3),
        )

    for product in products:
        print(product.model_dump())

# Run this cell when you have an internet connection.
asyncio.run(main())

## 7. Limit how many requests run at once (optional)

If an application launches hundreds of requests at the same time, it may overwhelm the service or hit rate limits. An `asyncio.Semaphore` can limit how many operations are active simultaneously.

You only need to understand the idea for now: **a semaphore is a concurrency limit.**

In [ ]:
async def limited_customer(customer_id, semaphore):
    async with semaphore:
        print(f"Starting customer {customer_id}")
        await asyncio.sleep(0.5)
        return f"Customer {customer_id}"

async def main():
    semaphore = asyncio.Semaphore(2)  # At most 2 tasks inside the protected section
    results = await asyncio.gather(
        *(limited_customer(customer_id, semaphore) for customer_id in range(1, 6))
    )
    print(results)

asyncio.run(main())

## 8. How async fits into AI engineering

Imagine an AI assistant needs to gather customer details, order status, and product information before composing a response.

A simplified flow could be:

1. Call independent APIs concurrently with HTTPX.
2. Validate responses with Pydantic.
3. Combine the validated results.
4. Pass the relevant context to an LLM or agent.

Async helps reduce idle waiting when the work involves multiple independent I/O calls. It does **not** guarantee that the LLM itself will generate tokens faster.

In [ ]:
# A small, offline example: gather three pieces of context concurrently.
async def get_customer_context():
    await asyncio.sleep(0.3)
    return {"customer": "Maya"}

async def get_order_context():
    await asyncio.sleep(0.3)
    return {"order_status": "shipped"}

async def get_product_context():
    await asyncio.sleep(0.3)
    return {"product": "Wireless headphones"}

async def build_support_context():
    customer, order, product = await asyncio.gather(
        get_customer_context(),
        get_order_context(),
        get_product_context(),
    )

    return {**customer, **order, **product}

asyncio.run(build_support_context())

## 9. Practice exercises

Try these before looking for help.

1. Change the sequential example to fetch five customers. What happens to the elapsed time?
2. Add a `get_invoice(invoice_id)` coroutine that waits for half a second and returns an invoice dictionary.
3. Use `asyncio.gather()` to fetch three invoices concurrently.
4. Make invoice 2 raise an exception. Handle the error so the other invoices still produce results.
5. Add a timeout to an operation that sleeps for two seconds.
6. In the HTTPX example, fetch products 4, 5, and 6, then validate each response with Pydantic.
7. **Challenge:** Use a semaphore to allow at most two simulated tasks to run inside the protected section at once.

### Quick knowledge check

- Does `async def` by itself make a function run concurrently? **No.**
- What does `await` do? **It waits for an awaitable result while allowing other async tasks to make progress.**
- When is async especially useful? **When the program spends time waiting for I/O.**
- Is async automatically faster for CPU-heavy calculations? **No.**
- Why reuse one HTTPX `AsyncClient`? **It allows connection reuse and avoids creating a new client for every request.**

## 10. Optional advanced topics — save for later

These are useful, but students do not need to master them in this introductory lesson:

- `asyncio.create_task()` — schedule a coroutine as a task.
- `asyncio.Queue` — coordinate producers and workers.
- `asyncio.to_thread()` — call blocking functions without blocking the event loop.
- Async database drivers such as `asyncpg` — database access using async I/O.
- Streaming LLM responses — process generated output as it arrives.

**Takeaway:** Learn `async def`, `await`, `asyncio.run()`, and `asyncio.gather()` first. Then practise asynchronous HTTP calls and response validation. Add queues, database pools, and more advanced concurrency patterns when building real applications.